# Barbet Harness（Colab 版）

驗證 Barbet 在 A／B／C 三種設定下的任務完成能力。

> ⚠️ 預設使用 **DUMMY 假模型**，結果不是 Barbet 的結果。取得主線的 checkpoint、renderer、解碼設定後，再到第 5 步切換。

## 1. 取得專案
從 GitHub 下載最新版本，直接執行即可。

In [ ]:
import os
if not os.path.exists('barbet-harness'):
    !git clone -q https://github.com/cathy910207-hub/barbet-harness.git
%cd barbet-harness
!git pull -q
!ls

## 2. 檢查流程
應顯示「全部檢查通過 ✓」。

In [ ]:
!python tests/test_pipeline.py

## 3. 用假模型劇本跑 A／B／C
劇本在 `tests/dummy_script.json`，用來確認各條路徑（修正成功、修正後變錯、編造來源……）都能被正確記錄。

In [ ]:
!python run_experiment.py --dummy-script tests/dummy_script.json

## 4. 查看報告與 trace

In [ ]:
import glob, json
from IPython.display import Markdown, display
run_dir = sorted(glob.glob('runs/*'))[-1]
display(Markdown(open(f'{run_dir}/summary.md', encoding='utf-8').read()))

In [ ]:
# 看某一題、某一組的完整 trace
TASK, GROUP = 'fix_001', 'C'
for line in open(f'{run_dir}/traces.jsonl', encoding='utf-8'):
    r = json.loads(line)
    if r['task_id'] == TASK and r['group'] == GROUP:
        for s in r['steps']:
            print('='*20, f"第 {s['step']+1} 步", '='*20)
            print(s['prompt'][-600:])
            print('--- 模型輸出 ---\n', s['output'])
            if s.get('tool_execution'): print('--- 工具結果 ---\n', s['tool_execution'])
            if s.get('runtime_error'): print('--- 回饋 ---\n', s['runtime_error']['message'])
        print('\n最終：', '成功' if r['success'] else f"失敗（{r['error_type']}）")

## 5. 切換到 Barbet（取得主線設定後）

1. 在 `config.json` 填入 `model.type = "barbet"`、`model.checkpoint`、`model.revision`、`model.label`
2. 填入 `tokenizer.name_or_path`
3. 把 `decoding` 改成主線指定的設定
4. 在 `harness/renderer.py` 加入主線的 renderer，並修改 `renderer.name`

先只跑一題確認可以運作：

In [ ]:
# !pip install -q transformers torch
# !python run_experiment.py --tasks doc_001 --groups A

## 6. 記憶位置與無關段落實驗
修改 `config.json` 的 `retrieval.position`（start／middle／end）與 `retrieval.num_distractors`，各跑一次並比較。

In [ ]:
import json, subprocess
RUN = False  # 接上 Barbet 後改成 True 才會真的執行
cfg = json.load(open('config.json', encoding='utf-8'))
for pos in ['start', 'middle', 'end']:
    cfg['retrieval'].update(position=pos, num_distractors=3)
    cfg['experiment_name'] = f'position_{pos}'
    json.dump(cfg, open(f'config_{pos}.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    if RUN:
        subprocess.run(['python', 'run_experiment.py', '--config', f'config_{pos}.json', '--groups', 'B,C'])
print('已產生 config_start/middle/end.json')

## 7. 下載結果

In [ ]:
from google.colab import files
!zip -qr runs.zip runs
files.download('runs.zip')